In [4]:
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser


In [5]:
load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b")


In [6]:
# Structure for critic evaluation
class QualityFeedback(BaseModel):
    score: int = Field(description="Quality score between 1 and 10 based on depth, clarity and accuracy")
    feedback: str = Field(description="Specific suggestions on what to improve, add or correct")

parser = PydanticOutputParser(pydantic_object=QualityFeedback)


In [7]:
# LangGraph state definition
class AnswerState(TypedDict):
    question: str
    draft_answer: str
    critique_feedback: str
    quality_score: int
    iteration: int


In [8]:
def generator_node(state: AnswerState):
    question = state["question"]
    feedback = state.get("critique_feedback", "")
    current_draft = state.get("draft_answer", "")
    current_iter = state.get("iteration", 0)

    if current_iter == 0 or not current_draft:
        # First attempt: generate initial draft
        prompt = f"""You are answering this question for a learner.
Question: {question}

Provide a clear, simple and informative explanation.
"""
    else:
        # Subsequent attempts: refine using critic feedback
        prompt = f"""You need to improve an earlier answer based on reviewer feedback.
Question: {question}

Previous Draft:
{current_draft}

Reviewer Feedback:
{feedback}

Please address each feedback point and write an improved, high-quality response.
"""

    response = llm.invoke(prompt)
    return {
        "draft_answer": response.content.strip(),
        "iteration": current_iter + 1
    }


In [9]:
def critic_node(state: AnswerState):
    prompt = PromptTemplate(
        template="""You are a strict quality critic.
Evaluate the following answer to the question.

Question: {question}
Answer: {answer}

Instructions:
- Score between 1 to 10 (be strict: basic or brief answers get 4-6, give 8+ only if comprehensive, accurate and well structured).
- Mention exactly what is missing or can be made clearer.

{format_instructions}
""",
        input_variables=["question", "answer"],
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )

    chain = prompt | llm | parser
    result = chain.invoke({
        "question": state["question"],
        "answer": state["draft_answer"]
    })

    return {
        "quality_score": result.score,
        "critique_feedback": result.feedback
    }


In [10]:
# Router: decide whether to refine again or stop
def should_continue(state: AnswerState) -> Literal["generator", END]:
    score = state.get("quality_score", 0)
    iter_count = state.get("iteration", 0)
    max_iterations = 3

    print(f"[Iteration {iter_count}] Score: {score}/10")

    # Stop if score is good enough or limit reached
    if score >= 8:
        print("Quality score is good (>=8). Done!")
        return END
    elif iter_count >= max_iterations:
        print("Hit max iteration limit. Stopping.")
        return END
    else:
        print("Feedback received. Sending back to generator to refine...")
        return "generator"


In [11]:
# Build the workflow graph
workflow = StateGraph(AnswerState)

workflow.add_node("generator", generator_node)
workflow.add_node("critic", critic_node)

workflow.add_edge(START, "generator")
workflow.add_edge("generator", "critic")

workflow.add_conditional_edges(
    "critic",
    should_continue,
    {
        "generator": "generator",
        END: END
    }
)

app = workflow.compile()


In [12]:
# Run the workflow
query = "Explain what is LangGraph and how it differs from LangChain?"

initial_state = {
    "question": query,
    "draft_answer": "",
    "critique_feedback": "",
    "quality_score": 0,
    "iteration": 0
}

final_result = app.invoke(initial_state)

print()
print(f"Total Iterations: {final_result.get('iteration')}")
print(f"Final Quality Score: {final_result.get('quality_score')}/10")
print("-" * 50)
print("Critic Feedback:")
print(final_result.get('critique_feedback'))
print("-" * 50)
print("Final Answer:")
print(final_result.get('draft_answer'))
